## Practice Problem: Subscriptions & Deliveries (40 ~ 60 Minutes)

### Scenario
You are analyzing data for a gourmet coffee subscription box company. 
You are handed two tables:
* `deliveries`: A log of individual box shipments sent to subscribers.
* `subscribers`: Customer profiles, including their sign-up city and plan tier.

### The Dataset

```python
import pandas as pd
import numpy as np

# 1. Deliveries Table (Contains strings for dates, messy missing values, and raw financial metrics)
deliveries = pd.DataFrame({
    'delivery_id': [9001, 9002, 9003, 9004, 9005, 9006, 9007, 9008],
    'subscriber_id': [101, 102, 101, 103, 104, 102, 105, 103],
    'coffee_type': ['Espresso Blend', 'Dark Roast', 'Espresso Blend', 
                     'Single Origin', 'Dark Roast', 'Single Origin', 'Espresso Blend', 'Single Origin'],
    'box_price': [45.0, 30.0, 45.0, np.nan, 30.0, 55.0, 45.0, 55.0],  # One missing value to handle!
    'ship_date': ['2026-05-01', '2026-05-01', '2026-05-02', '2026-05-02', 
                  '2026-05-03', '2026-05-03', '2026-05-04', '2026-05-04']
})

# 2. Subscribers Table (Metadata for merging)
subscribers = pd.DataFrame({
    'subscriber_id': [101, 102, 103, 104, 105],
    'city': ['London', 'Toronto', 'London', 'Vancouver', 'Toronto'],
    'tier': ['Premium', 'Standard', 'Premium', 'Standard', 'Premium']
})

```

### Questions

1. **Data Inspection & Cleaning**
* Inspect both DataFrames using `info()`.
* Convert the `ship_date` column in `deliveries` to a proper datetime data type.
* Identify the missing value in `box_price` and fill it with the overall average box price.


2. **Feature Engineering**
* Create a new column called `shipping_cost` that is equal to 8% of the `box_price`.
* Create a boolean column called `is_premium_coffee` that returns `True` if the `box_price` is $45.0 or higher, and `False` otherwise.


3. **GroupBy Aggregations**
* Find the total revenue (`box_price` sum), the total number of boxes shipped, and the average box price for each unique `coffee_type`.


4. **Table Merging & Segment Analysis**
* Merge the `deliveries` and `subscribers` DataFrames using a left join on `subscriber_id`.
* Using this newly merged dataset, calculate both the total revenue and the average box price broken down by customer `tier`.


5. **Geographic Performance**
* Group the merged dataset by `city` to calculate the average box spend for each city, and sort the results from highest to lowest.


6. **Time Series & Anomaly Detection**
* Calculate the total revenue generated for each daily `ship_date`.
* Apply a rolling window to calculate a 2-day rolling average of daily revenue.
* Identify any "busy days" where the day's total revenue exceeded the baseline daily average by more than 0.5 standard deviations.


7. **Pivot Tables**
* Generate a clean, report-friendly pivot table showing the total revenue, using `city` as the row index and `coffee_type` as the column headers. Ensure missing values are filled with `0`.



```


# Dataset

In [1]:
import pandas as pd
import numpy as np

# 1. Deliveries Table (Contains strings for dates, messy missing values, and raw financial metrics)
deliveries = pd.DataFrame({
    'delivery_id': [9001, 9002, 9003, 9004, 9005, 9006, 9007, 9008],
    'subscriber_id': [101, 102, 101, 103, 104, 102, 105, 103],
    'coffee_type': ['Espresso Blend', 'Dark Roast', 'Espresso Blend', 
                     'Single Origin', 'Dark Roast', 'Single Origin', 'Espresso Blend', 'Single Origin'],
    'box_price': [45.0, 30.0, 45.0, np.nan, 30.0, 55.0, 45.0, 55.0],  # One missing value to handle!
    'ship_date': ['2026-05-01', '2026-05-01', '2026-05-02', '2026-05-02', 
                  '2026-05-03', '2026-05-03', '2026-05-04', '2026-05-04']
})

# 2. Subscribers Table (Metadata for merging)
subscribers = pd.DataFrame({
    'subscriber_id': [101, 102, 103, 104, 105],
    'city': ['London', 'Toronto', 'London', 'Vancouver', 'Toronto'],
    'tier': ['Premium', 'Standard', 'Premium', 'Standard', 'Premium']
})

### 1. Data Inspection & Cleaning

In [2]:
deliveries.info()
subscribers.info()
deliveries["ship_date"] = pd.to_datetime(deliveries["ship_date"])
avg_price = deliveries["box_price"].mean()
deliveries["box_price"] = deliveries["box_price"].fillna(avg_price)
print("Cleaned Box Prices:\n", deliveries[['delivery_id', 'box_price']])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   delivery_id    8 non-null      int64  
 1   subscriber_id  8 non-null      int64  
 2   coffee_type    8 non-null      object 
 3   box_price      7 non-null      float64
 4   ship_date      8 non-null      object 
dtypes: float64(1), int64(2), object(2)
memory usage: 448.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   subscriber_id  5 non-null      int64 
 1   city           5 non-null      object
 2   tier           5 non-null      object
dtypes: int64(1), object(2)
memory usage: 248.0+ bytes
Cleaned Box Prices:
    delivery_id  box_price
0         9001  45.000000
1         9002  30.000000
2         9003  45.000000
3         9004  43.571429
4   

### 2. Feature Engineering

In [3]:
deliveries["shipping_cost"] = deliveries["box_price"] * 0.08
deliveries["is_premium_coffee"] = deliveries["box_price"] >= 45
print(deliveries.head(3))

   delivery_id  subscriber_id     coffee_type  box_price  ship_date  \
0         9001            101  Espresso Blend       45.0 2026-05-01   
1         9002            102      Dark Roast       30.0 2026-05-01   
2         9003            101  Espresso Blend       45.0 2026-05-02   

   shipping_cost  is_premium_coffee  
0            3.6               True  
1            2.4              False  
2            3.6               True  


### 3. GroupBy Aggregations

In [4]:
print(deliveries.groupby("coffee_type")["box_price"].agg(["sum","count","mean"]))

                       sum  count       mean
coffee_type                                 
Dark Roast       60.000000      2  30.000000
Espresso Blend  135.000000      3  45.000000
Single Origin   153.571429      3  51.190476


### 4. Table Merging & Segment Analysis

In [5]:
full_data = deliveries.merge(subscribers, on="subscriber_id", how="left")
print(full_data.groupby("tier")["box_price"].agg(["sum", "mean"]))

                 sum       mean
tier                           
Premium   233.571429  46.714286
Standard  115.000000  38.333333


### 5. Geographic Performance

In [6]:
print(full_data.groupby("city")["box_price"].mean().sort_values(ascending=False))

city
London       47.142857
Toronto      43.333333
Vancouver    30.000000
Name: box_price, dtype: float64


### 6. Time Series & Anomaly Detection

In [7]:
daily_revenue = full_data.groupby("ship_date")["box_price"].sum()
print(daily_revenue)
rolling_avg = daily_revenue.rolling(2).mean()
print(rolling_avg)
baseline_mean = daily_revenue.mean()
baseline_std = daily_revenue.std()
busy_days = daily_revenue[daily_revenue > baseline_mean + (0.5 * baseline_std)]
print(busy_days)

ship_date
2026-05-01     75.000000
2026-05-02     88.571429
2026-05-03     85.000000
2026-05-04    100.000000
Name: box_price, dtype: float64
ship_date
2026-05-01          NaN
2026-05-02    81.785714
2026-05-03    86.785714
2026-05-04    92.500000
Name: box_price, dtype: float64
ship_date
2026-05-04    100.0
Name: box_price, dtype: float64


### 7. Pivot Tables

In [8]:
pivot = full_data.pivot_table(values="box_price",index="city",columns="coffee_type",aggfunc="sum",fill_value=0)
print(pivot)

coffee_type  Dark Roast  Espresso Blend  Single Origin
city                                                  
London              0.0            90.0      98.571429
Toronto            30.0            45.0      55.000000
Vancouver          30.0             0.0       0.000000
